In [1]:
from pycaret.classification import setup, create_model, finalize_model, save_model, predict_model

In [2]:
# Import dataset
import pandas as pd
df = pd.read_csv('Dataset_LST_Massive_20k.csv')
df.head()

,Region_Type,Latitude,Longitude,Altitude_m,Day_of_year,Ndvi,Albedo,Cloud_cover_pct,Humidity_pct,Insolation_wm2,Lst_celsius
0,Dense_Urban_Temperate,48.8722,2.2984,68,0,0.0384,0.0970,20.78,50,200,10.23
1,Dense_Urban_Temperate,48.8802,2.3543,62,0,0.0196,0.0926,41.81,50,200,9.30
2,Dense_Urban_Temperate,48.8444,2.3000,43,0,0.0535,0.1158,20.33,50,200,10.36
3,Dense_Urban_Temperate,48.8535,2.3226,40,0,0.0757,0.1049,34.26,50,200,9.87
4,Dense_Urban_Temperate,48.8202,2.3462,64,0,0.1615,0.0870,39.36,50,200,8.06


In [4]:
# Encodage cyclique de la date
import numpy as np

df["day_sin"] = np.sin(2*np.pi*df["Day_of_year"]/365)

df["day_cos"] = np.cos(2*np.pi*df["Day_of_year"]/365)

df.drop(columns=["Day_of_year"], inplace=True)

In [ ]:

# 1. Préparation des données
colonnes_utiles_clf = ['Latitude', 'Longitude', 'Altitude_m', 'Ndvi', 'Albedo', 
                       'Cloud_cover_pct', 'Lst_celsius', 'day_sin', 'day_cos', 'Region_Type']
df_pycaret_clf = df[colonnes_utiles_clf]

# 2. Initialisation PyCaret
# fix_imbalance=True est magique : il équilibre les classes automatiquement en arrière-plan
clf_experiment = setup(data=df_pycaret_clf, 
                       target='Region_Type', 
                       normalize=True, 
                       normalize_method='zscore',
                       fix_imbalance=True, 
                       session_id=42)

,Description,Value
0,Session id,42
1,Target,Region_Type
2,Target type,Multiclass
3,Target mapping,"Arid_Sandy_Desert: 0, Coastal_Oceanic_Forest: 1, Continental_Boreal_Taiga: 2, Dense_Urban_Temperate: 3, Equatorial_Wetland_Basin: 4, High_Altitude_Alpine: 5, Hyper_Arid_Desert_Floor: 6, Intensive_Agricultural_Plain: 7, Polar_Coastal_Desert: 8, Polar_Ice_Sheet: 9, Subtropical_Urban_Canopy: 10, Tropical_Rainforest_Canopy: 11"
4,Original data shape,"(16779, 10)"
5,Transformed data shape,"(18342, 10)"
6,Transformed train set shape,"(13308, 10)"
7,Transformed test set shape,"(5034, 10)"
8,Numeric features,9
9,Preprocess,True


In [7]:
# 3. Création du modèle Random Forest avec Validation Croisée (5 folds)
print("Création du modèle et Validation Croisée...")
cv_clf_model = create_model('rf', fold=5)

Création du modèle et Validation Croisée...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
3,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
4,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
Mean,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
Std,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [ ]:
# 4. Finalisation et Sauvegarde
final_clf_model = finalize_model(cv_clf_model)
save_model(final_clf_model, "best_region_classifier_pycaret")

print("Pipeline PyCaret sauvegardé. (PyCaret intègre l'encodeur de labels directement dans son pipeline !)")